# 📚 Build a Novel Agent — LangChain · LangGraph · LangSmith

In this 2-hour hands-on workshop you'll build an AI agent that **reads a whole novel** and holds a
multi-turn conversation about its characters, events, chapters, relationships, themes and ending.

| Module | You'll learn | Time |
|---|---|---|
| 0. Setup | keys, config, pre-flight check | 10 min |
| 1. Ingestion | load → clean → chapters → chunks → embeddings → Chroma | 20 min |
| 2. Enrichment | LLM map-reduce with structured output: chapter notes, characters, relationships, themes | 15 min |
| 3. Retrieval lab | dense vs BM25 vs hybrid (RRF) vs reranked; chapter filters | 15 min |
| 4. The agent | LangGraph state, nodes, tools, the plan → act loop, tracing | 25 min |
| 5. Memory | threads (checkpointer), summaries, long-term store & the spoiler guard | 15 min |
| 6. Evaluate | LangSmith datasets, LLM-as-judge, experiments; Streamlit UI | 15 min |

```
INGEST                                                   QUERY (LangGraph)
book.txt → chapters → chunks → Voyage embeddings → Chroma    START → manage_memory → analyze → agent ⇄ tools → END
                  └→ Gemini map-reduce → knowledge.json       memory: SqliteSaver (threads) + Store (reader profile)
                                                               retrieval: dense + BM25 → RRF → Voyage rerank
```

All the code lives in the `novel_agent/` package — this notebook calls into it step by step so you can
see (and change) what each piece does. Cells marked **✏️ Try it** are exercises.

## 0 · Setup

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display
from novel_agent.config import get_settings

BOOK_ID = "pride_and_prejudice"
settings = get_settings()
print("agent model :", settings.agent_model)
print("enrich model:", settings.enrich_model)
print("embeddings  :", settings.embeddings_provider, settings.embed_model, "| rerank:", settings.rerank_enabled)

Run the pre-flight check. Every line should say `[ OK ]` (fix your `.env` otherwise).

In [ ]:
!"{sys.executable}" -m novel_agent.doctor

## 1 · Ingestion: from a raw text file to a searchable index

Real books are messy. Project Gutenberg's *Pride and Prejudice* (#1342) has licence boilerplate, a 5,000-word
preface, a list of illustrations and 155 `[Illustration: …]` blocks — and chapter 1's heading is hidden at the
end of one of them (`Chapter I.]`). Garbage in, garbage out: cleaning is part of RAG.

In [ ]:
from novel_agent.books import PRESETS
from novel_agent.loaders import load_book

preset = PRESETS[BOOK_ID]
raw = load_book(gutenberg_id=preset.gutenberg_id, title=preset.title, author=preset.author)
print(f"{raw.title} by {raw.author}: {len(raw.text):,} characters")
print(raw.text[:1200])

In [ ]:
from novel_agent.chapters import split_chapters

chapters = split_chapters(raw.text, preset.chapter_pattern)
print(len(chapters), "chapters,", sum(c.word_count for c in chapters), "words")
for c in chapters[:5]:
    print(f"#{c.number:<3} {c.heading:<14} {c.word_count:>5} words | {c.text[:70]!r}")

Chapters are the backbone of everything that follows: each chunk, summary and citation carries its chapter
number, which is what makes *"what happens in chapter 34?"*, chronological evidence and the spoiler guard possible.

Now split each chapter into **chunks** small enough to embed precisely, with overlap so sentences at a boundary
aren't lost. Each chunk gets a contextual header (`[Pride and Prejudice · Chapter 34]`).

In [ ]:
from novel_agent.chunking import chunk_chapters

chunks = chunk_chapters(chapters, book_id=BOOK_ID, title=raw.title,
                        chunk_size=settings.chunk_size, chunk_overlap=settings.chunk_overlap)
print(len(chunks), "chunks")
sample = chunks[400]
print(sample.metadata)
print(sample.page_content)

**✏️ Try it:** re-run the cell above with `chunk_size=500` and `chunk_size=3000`. How does the number of chunks change?
What would you expect to happen to retrieval precision vs. context for each?

### Embeddings + vector store

An embedding model maps text to a vector so that *similar meaning ⇒ nearby vectors*. We use **Voyage AI**
(`voyage-4`) and store vectors in a persistent **Chroma** collection on disk.

In [ ]:
from novel_agent.config import get_embeddings

embeddings = get_embeddings()
vector = embeddings.embed_query("Mr. Darcy's first proposal")
print(len(vector), "dimensions:", [round(x, 3) for x in vector[:6]], "...")

Now run the ingestion pipeline up to the vector index (enrichment comes in Module 2). It is resumable — re-running skips chunks that are already embedded.

In [ ]:
from novel_agent.ingest import main as ingest

ingest(["--book", BOOK_ID, "--skip-enrich"])

## 2 · Enrichment: teaching the index about the *story*

Chunk retrieval is great for details (*"what did Darcy's letter say?"*) but no single chunk answers
*"what are the themes?"* or *"how does Elizabeth change?"*. So we run an LLM **map-reduce** once at ingestion:

- **map**: batches of chapters → `ChapterNotes` (summary, key events, characters, relationships, themes)
- **reduce**: all notes → `BookProfile` (synopsis, ending, characters with aliases & arcs, themes)
- **python**: resolve aliases (*Lizzy* = *Eliza* = *Elizabeth Bennet*), per-character appearances, relationship timelines

`with_structured_output(PydanticModel)` makes Gemini return validated objects instead of free text.

In [ ]:
from novel_agent.config import get_llm
from novel_agent.enrich import summarize_chapters

notes = summarize_chapters(chapters[:2], title=raw.title, author=raw.author,
                           llm=get_llm("enrich"), chapters_per_call=2)
print(json.dumps(notes[0].model_dump(), indent=2, ensure_ascii=False)[:2500])

Now enrich the whole book (~9 Gemini calls) and embed the chapter summaries. The result is cached in `data/index/<book>/knowledge.json`.

In [ ]:
ingest(["--book", BOOK_ID])

In [ ]:
from novel_agent.indexing import index_paths
from novel_agent.knowledge import BookKnowledge

knowledge = BookKnowledge.load(index_paths(BOOK_ID).knowledge)
print(knowledge.character_profile("Lizzy"))

In [ ]:
print(knowledge.relationship("Elizabeth", "Darcy"))

**✏️ Try it:** inspect `knowledge.alias_map`. Are any aliases wrong or ambiguous? What does
`knowledge.resolve_character("Miss Bennet")` return, and why is that name tricky in this novel?

## 3 · Retrieval lab: dense, sparse, hybrid, reranked

- **Dense** (embeddings) understands meaning and paraphrase.
- **Sparse** (BM25 keyword scoring) nails rare exact tokens — character and place names.
- **Hybrid** fuses both ranked lists with **Reciprocal Rank Fusion**: `score = Σ 1/(60 + rank)`.
- **Rerank**: a cross-encoder (Voyage `rerank-3-lite`) re-scores the top candidates against the query.

In [ ]:
from novel_agent.resources import load_resources

res = load_resources(BOOK_ID)

def show(title, docs, n=5):
    print(f"--- {title}")
    for d in docs[:n]:
        text = d.page_content.split("\n", 1)[-1].replace("\n", " ")
        print(f"  Ch {d.metadata['chapter']:>2} | {text[:95]}")

query = "What lies did Wickham tell about Darcy?"
show("dense", res.chunks.dense(query, k=5))
show("bm25", res.chunks.sparse(query, k=5))
show("hybrid (RRF)", res.chunks.hybrid(query, k=5))
show("hybrid + rerank", res.chunks.search(query, k=5))

In [ ]:
query = "the moment she realises she has been blind and prejudiced"
show("dense", res.chunks.dense(query, k=5))
show("bm25", res.chunks.sparse(query, k=5))
show("hybrid + rerank", res.chunks.search(query, k=5))

Metadata filters restrict search to part of the book — the basis of the spoiler guard:

In [ ]:
show("'proposal' in chapters 1-20", res.chunks.search("marriage proposal", k=4, chapter_to=20))
show("'proposal' in chapters 30-61", res.chunks.search("marriage proposal", k=4, chapter_from=30))

**✏️ Try it:** find one query where BM25 clearly beats dense retrieval, and one where dense wins.
(Hint: exact names like *Pemberley* or *Rosings* vs. descriptions of feelings.)

## 4 · The agent (LangGraph)

The agent is a `StateGraph` over `AgentState` (messages + running summary + plan). Nodes:

1. **manage_memory** – keeps long threads small (Module 5)
2. **analyze** – *structured output*: rewrites the question as standalone, classifies it and writes 1-4 research steps
3. **agent** – Gemini with 8 tools; decides which tool(s) to call next, or answers with `[Ch. N]` citations
4. **tools** – `ToolNode` runs the calls; results go back to the agent → the **multi-step reasoning loop**

In [ ]:
from novel_agent.tools import make_tools

for t in make_tools(res):
    print(f"{t.name:<24} {t.description.split(chr(10))[0][:95]}")

In [ ]:
from novel_agent.graph import build_graph
from novel_agent.memory import in_memory

checkpointer, store = in_memory()
graph = build_graph(res, checkpointer=checkpointer, store=store)
print(graph.get_graph().draw_mermaid())

In [ ]:
try:  # renders via the mermaid.ink web service
    from IPython.display import Image
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Diagram rendering unavailable:", exc)

In [ ]:
from novel_agent.runner import stream_turn

def chat(question, thread_id="demo", user_id="workshop-user", steps=True):
    print(f"🧑 {question}")
    for ev in stream_turn(graph, question, thread_id=thread_id, user_id=user_id, book_id=BOOK_ID):
        kind = ev["type"]
        if kind == "plan" and steps:
            a = ev["analysis"]
            print(f"  🧭 [{a['question_type']}] {a['standalone_question']}")
            for s in a["sub_questions"]:
                print(f"      • {s}")
        elif kind == "tool_call" and steps:
            print(f"  🔧 {ev['name']}({', '.join(f'{k}={v!r}' for k, v in ev['args'].items())})")
        elif kind == "memory" and steps:
            print("  🧠 older turns folded into the summary")
        elif kind == "final":
            display(Markdown(ev["text"]))

chat("How does Elizabeth's opinion of Darcy change over the course of the novel?")

Open **LangSmith** → project `novel-agent-workshop` → the latest `novel_agent_turn` trace. You can see every node,
the exact prompts, each tool call and its output, the `hybrid_search` retriever runs, token usage and latency.

In [ ]:
chat("What are the main themes of the novel, and which chapters develop them most?")

**✏️ Try it:**
1. Ask about something that is *not* in the book (e.g. *"What happens to Elizabeth after she has children?"*). Does it admit it?
2. Edit the system prompt, rebuild and compare: `import novel_agent.prompts as p; p.AGENT_SYSTEM += "\nAlways end with one discussion question for the reader."` then re-run the `build_graph` cell.

## 5 · Memory

**Short-term (thread) memory** — the checkpointer saves the graph state after every step, per `thread_id`.
Follow-ups work because the analyzer sees the conversation and rewrites *"her"* / *"that"* into a standalone question.

In [ ]:
chat("What made her reconsider?")  # same thread as before: 'her' = Elizabeth, 'reconsider' = Darcy

In [ ]:
config = {"configurable": {"thread_id": "demo"}}
state = graph.get_state(config)
print("messages in thread:", len(state.values["messages"]))
print("checkpoints (time-travel history):", len(list(graph.get_state_history(config))))
print("last plan:", state.values["analysis"])

In [ ]:
chat("What made her reconsider?", thread_id="fresh-thread")  # new thread = no context

**Long-term memory** — the LangGraph **store** keeps a *reader profile* per `user_id`, shared across all threads.
Tell the agent how far you've read; the tools then filter out later chapters (the spoiler guard is enforced in code,
not just asked for in the prompt).

In [ ]:
chat("I've only read up to chapter 20. What do you make of Mr. Wickham so far?", thread_id="alice-1", user_id="alice")

In [ ]:
from novel_agent import memory
print(memory.get_reader_profile(store, "alice"))
chat("How does the Lydia and Wickham storyline end?", thread_id="alice-2", user_id="alice")  # NEW thread, same reader

**Summarization** — long threads would eventually overflow the context window (and cost more every turn).
After `MAX_TURNS_BEFORE_SUMMARY` questions, `manage_memory` folds older turns into `state["summary"]` and removes
them with `RemoveMessage` (whole turns only, so tool calls and results stay paired).

**✏️ Try it:** make it trigger quickly and watch it happen.

In [ ]:
import novel_agent.graph as g
g.MAX_TURNS_BEFORE_SUMMARY, g.KEEP_TURNS = 2, 1
for q in ["Who is Jane Bennet?", "Who does she love?", "Does it end well for them?"]:
    chat(q, thread_id="summary-demo", steps=False)
values = graph.get_state({"configurable": {"thread_id": "summary-demo"}}).values
print("SUMMARY:", values["summary"])
print("messages kept:", len(values["messages"]))
g.MAX_TURNS_BEFORE_SUMMARY, g.KEEP_TURNS = 6, 3

## 6 · Evaluate with LangSmith

"It looked good on three questions" is not an evaluation. `eval/questions.jsonl` has 14 reference Q&As across
every question type (including multi-turn follow-ups). `eval/run_eval.py`:

1. uploads them as a LangSmith **dataset**,
2. runs the agent on each example (fresh thread per example),
3. scores each answer with an **LLM-as-judge** (correctness vs. reference) plus heuristics (*cites a chapter?*, *used a tool?*),
4. records everything as an **experiment** you can compare side by side in the LangSmith UI.

In [ ]:
!"{sys.executable}" eval/run_eval.py --limit 4

**✏️ Try it:** change one thing — e.g. `RERANK_ENABLED=false`, `AGENT_MODEL=gemini-3.8-flash` or a prompt tweak —
re-run the eval, and compare the two experiments in LangSmith (*Datasets → novel-agent-pride_and_prejudice → Experiments*).

### The chat UI

In a terminal (with the venv activated):

```
streamlit run app.py
```

It shows the plan and tool calls live, the source passages behind each answer, a spoiler-guard slider, and 👍/👎
buttons that attach feedback to the LangSmith trace.

## 🎉 Wrap-up & where to go next

You built: a cleaning + chapter-aware ingestion pipeline · hybrid retrieval with RRF and reranking · an LLM
map-reduce knowledge layer · a LangGraph agent with a planning step and 8 tools · thread memory, summarization and a
long-term reader profile · tracing, a dataset and an LLM-judged evaluation in LangSmith.

**Extensions**
- Add a **self-check node** after `agent` that verifies every `[Ch. N]` citation against the retrieved passages.
- Turn the relationship notes into a **graph** (GraphRAG) and add a `path_between(a, b)` tool.
- Ingest another book: `python -m novel_agent.ingest --book frankenstein` or `--file your_book.epub`.
- Serve the graph with `langgraph dev` and explore it in LangGraph Studio.
- Add **human-in-the-loop**: `interrupt()` before `set_reading_progress` to confirm with the reader.